In [ ]:
# Parameters contain references only, never connection strings or host/database values.
F12_NONINTERACTIVE = False
F12_EVENTSTREAM_ID = ""
F12_KQL_DATABASE_ID = ""
F12_VAULT_URL = ""
F12_SECRET_NAME = ""
F12_WHEEL_PAYLOADS = {}  # Agent renderer injects public wheel bytes here.

import contextlib, io, logging, json
_F12_RECORDS = []
def _f12_record(value):
    _F12_RECORDS.append(value)
    print(json.dumps(value), flush=True)

@contextlib.contextmanager
def _f12_silent():
    previous = logging.root.manager.disable
    try:
        logging.disable(logging.CRITICAL)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(previous)

import sys, platform, json, importlib.metadata, os, pathlib
_f12_record({"stage":"before_install", "python":platform.python_version(),"platform_system":platform.system(),"machine":platform.machine(),"cpu_count":os.cpu_count(),"packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow")}})
for path in ("/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/memory.max"):
    if pathlib.Path(path).exists(): _f12_record({"stage":"resource_limit", "path":path,"value":pathlib.Path(path).read_text().strip()})

_F12_WORKER_SOURCE = '"""Fresh interpreter for notebook workloads; private pipe protocol, no secrets on disk.\nThe parent notebook owns authentication. Token/secret/prompt responses only traverse\nstdin; stdout contains private protocol frames that the parent filters before display.\n"""\n\nimport base64\nimport contextlib\nimport io\nimport json\nimport logging\nimport sys\nimport time\nimport types\n\nWORKSPACE = "e90940bd-3f32-42ce-998e-ef2ddc74355b"\n_PIPE_OUT = sys.stdout\n_PIPE_IN = sys.stdin\n\n\ndef request(action, **fields):\n    _PIPE_OUT.write(json.dumps({"rpc": action, **fields}) + "\\n")\n    _PIPE_OUT.flush()\n    response = json.loads(_PIPE_IN.readline())\n    if response.get("ok") is not True:\n        raise RuntimeError("NotebookBridgeUnavailable")\n    return response["value"]\n\n\ndef record(value):\n    _PIPE_OUT.write(json.dumps({"record": value}) + "\\n")\n    _PIPE_OUT.flush()\n\n\n@contextlib.contextmanager\ndef quiet():\n    previous = logging.root.manager.disable\n    try:\n        logging.disable(logging.CRITICAL)\n        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\n            yield\n    finally:\n        logging.disable(previous)\n\n\nclass NotebookCredential:\n    """Same notebook identity; refresh through the parent, never ambient SP auth."""\n\n    def get_token(self, *scopes, **kwargs):\n        permitted = {\n            "https://analysis.windows.net/powerbi/api/.default",\n            "https://api.fabric.microsoft.com/.default",\n        }\n        if not scopes or any(scope not in permitted for scope in scopes):\n            raise RuntimeError("CredentialScopeNotAllowed")\n        raw = request("token", audience="pbi")\n        payload = raw.split(".")[1]\n        expiry = int(\n            json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))["exp"]\n        )\n        if expiry <= time.time():\n            raise RuntimeError("NotebookTokenExpired")\n        from azure.core.credentials import AccessToken\n\n        return AccessToken(raw, expiry)\n\n\ndef dependency_check(task):\n    import importlib.metadata\n\n    import numpy\n    import pandas\n    import pyarrow\n\n    import shape\n\n    versions = {\n        "numpy": numpy.__version__,\n        "pandas": pandas.__version__,\n        "pyarrow": pyarrow.__version__,\n    }\n    matches = all(importlib.metadata.version(name) == value for name, value in versions.items())\n    record(\n        {\n            "stage": "worker_runtime",\n            "task": task,\n            "fresh_interpreter": True,\n            "python": sys.version.split()[0],\n            "shape": shape.__version__,\n            "loaded_versions": versions,\n            "versions_match": matches,\n        }\n    )\n    if not matches or numpy.__version__.split(".")[0] != "2" or shape.__version__ != "0.9.4":\n        raise RuntimeError("FreshDependencyMismatch")\n\n\ndef main():\n    envelope = json.loads(_PIPE_IN.readline())\n    task = envelope["task"]\n    config = envelope["config"]\n    source = envelope["source"]\n    if task not in ("after_install", "memory", "semantic_model", "events"):\n        raise RuntimeError("WorkerTaskNotAllowed")\n    credentials = types.SimpleNamespace(\n        getToken=lambda audience: request("token", audience=audience),\n        getSecret=lambda vault, name: (\n            request("secret")\n            if vault == config["F12_VAULT_URL"] and name == config["F12_SECRET_NAME"]\n            else (_ for _ in ()).throw(RuntimeError("SecretReferenceMismatch"))\n        ),\n    )\n    bridge = types.SimpleNamespace(\n        credentials=credentials, notebook=types.SimpleNamespace(exit=lambda _: None)\n    )\n    namespace = {\n        **config,\n        "F12_INSTALL_OK": True,\n        "_f12_record": record,\n        "_f12_silent": quiet,\n        "_f12_notebook_bridge": bridge,\n        "contextlib": contextlib,\n        "io": io,\n    }\n    try:\n        with quiet(), contextlib.ExitStack() as stack:\n            dependency_check(task)\n            if task in ("semantic_model", "events"):\n                # Set context/credential before importing sempy. No SP fixture is activated.\n                from fabric.analytics.environment.context import SetFabricDefaultContext\n                from fabric.analytics.environment.credentials import (\n                    SetFabricAnalyticsDefaultTokenCredentials,\n                )\n\n                stack.enter_context(SetFabricDefaultContext(workspace_id=WORKSPACE))\n                stack.enter_context(SetFabricAnalyticsDefaultTokenCredentials(NotebookCredential()))\n            if task == "events":\n                import getpass\n\n                getpass.getpass = lambda label: request("prompt", label=label)\n            exec(compile(source, "<f12-fresh-workload>", "exec"), namespace)\n        return 0\n    except BaseException as error:\n        # pytest Failed is a BaseException; user cancellation still terminates this worker.\n        record({"stage": task, "result": "failed", "error_type": type(error).__name__})\n        return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
# Parent notebook: standard library only; keep its cached NumPy out of workloads.
import math, re, subprocess, warnings, getpass

class ObservedFailure(Exception):
    pass

def safe_records(raw):
    """Allow only expected measurement fields from our own structured exit value."""
    records = json.loads(raw)
    if not isinstance(records, list):
        raise ObservedFailure("exit_value_shape")
    stages = {
        "before_install",
        "after_install",
        "resource_limit",
        "wheel_install",
        "generation",
        "profile",
        "child_exit",
        "error",
        "memory",
        "semantic_model",
        "destination_readiness",
        "events_sent",
        "events_landed",
        "events",
        "worker_runtime",
    }
    numeric = {
        "cpu_count",
        "rows",
        "columns",
        "seconds",
        "parquet_bytes",
        "peak_rss_raw",
        "exit_code",
        "relationships",
        "emitter_acknowledged",
        "stream_writer_acknowledged",
        "direct_writer_accepted",
        "poll_attempts",
        "poll_seconds",
    }
    flags = {
        "hashes_verified",
        "installed",
        "source_running",
        "destination_running",
        "warning",
        "counts_match",
        "versions_match",
        "fresh_interpreter",
    }
    enums = {
        "task": {"after_install", "memory", "semantic_model", "events"},
        "result": {"failed", "skipped"},
        "kernel": {"python", "rust"},
        "mode": {"bounded"},
        "platform_system": {"Linux", "Windows", "Darwin"},
        "machine": {"x86_64", "aarch64", "AMD64", "arm64"},
        "path": {"/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/memory.max"},
        "peak_rss_unit": {"KiB", "bytes"},
        "destination_status": {
            "NotChecked",
            "Creating",
            "Running",
            "Warning",
            "Stopped",
            "Paused",
            "Starting",
            "Failed",
            "Other",
        },
        "imports": {"passed"},
        "test_body": {"passed", "failed"},
    }
    output = []
    for record in records:
        if not isinstance(record, dict) or record.get("stage") not in stages:
            raise ObservedFailure("exit_value_record_shape")
        clean = {"stage": record["stage"]}
        for key, value in record.items():
            if key in numeric and type(value) in (int, float) and math.isfinite(value):
                clean[key] = value
            elif key in flags and type(value) is bool:
                clean[key] = value
            elif key in enums and isinstance(value, str) and value in enums[key]:
                clean[key] = value
            elif (
                key in ("python", "shape")
                and isinstance(value, str)
                and re.fullmatch(r"[0-9][0-9A-Za-z.+-]{0,50}", value)
            ):
                clean[key] = value
            elif (
                key == "error_type"
                and isinstance(value, str)
                and re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,70}", value)
            ):
                clean[key] = value
            elif (
                key == "value"
                and record["stage"] == "resource_limit"
                and isinstance(value, str)
                and re.fullmatch(r"(?:max|[0-9]+)(?: [0-9]+)?", value)
            ):
                clean[key] = value
            elif key in ("packages", "loaded_versions") and isinstance(value, dict):
                clean[key] = {
                    k: v
                    for k, v in value.items()
                    if k in ("numpy", "pandas", "pyarrow", "semantic-link-sempy")
                    and isinstance(v, str)
                    and re.fullmatch(r"[0-9][0-9A-Za-z.+-]{0,50}", v)
                }
            elif key in ("raw_rows_by_table", "unique_rows_by_table") and isinstance(value, list):
                if all(
                    isinstance(row, list)
                    and len(row) == 2
                    and row[0] in ("f12-emitter", "f12-writer")
                    and type(row[1]) is int
                    for row in value
                ):
                    clean[key] = value
            elif key == "direct_writer_count" and isinstance(value, list):
                if all(
                    isinstance(row, list) and len(row) == 1 and type(row[0]) is int for row in value
                ):
                    clean[key] = value
            elif key == "tables" and isinstance(value, list):
                clean[key] = [
                    {
                        "table_index": i,
                        **{
                            k: v
                            for k, v in row.items()
                            if k in ("rows", "columns") and type(v) is int
                        },
                    }
                    for i, row in enumerate(value)
                    if isinstance(row, dict)
                ]
        output.append(clean)
    return output

_F12_PROMPTS = {
    "f12-input connection string (hidden): ",
    "New f12 Eventhouse Query URI host (hidden): ",
    "New f12 KQL database name (hidden): ",
    "Namespace copied from this f12-input Details (hidden): ",
    "Event hub name copied from this f12-input Details (hidden): ",
}

def _f12_bridge_answer(frame):
    try:
        action = frame.get("rpc")
        if action == "prompt":
            if F12_NONINTERACTIVE or frame.get("label") not in _F12_PROMPTS:
                raise RuntimeError("PromptNotAllowed")
            with warnings.catch_warnings():
                warnings.simplefilter("error", getpass.GetPassWarning)
                value = getpass.getpass(frame["label"])
        else:
            with _f12_silent():
                import notebookutils
                if action == "token" and frame.get("audience") in ("pbi", "kusto"):
                    value = notebookutils.credentials.getToken(frame["audience"])
                elif action == "secret" and F12_VAULT_URL and F12_SECRET_NAME:
                    value = notebookutils.credentials.getSecret(F12_VAULT_URL, F12_SECRET_NAME)
                else:
                    raise RuntimeError("BridgeActionNotAllowed")
        return {"ok": True, "value": value}
    except Exception:
        return {"ok": False}  # Never return SDK diagnostics, names or connection values.


def _f12_child(task, source):
    config = {name: globals()[name] for name in (
        "F12_NONINTERACTIVE", "F12_EVENTSTREAM_ID", "F12_KQL_DATABASE_ID",
        "F12_VAULT_URL", "F12_SECRET_NAME")}
    proc = None
    observed = []
    try:
        # All tokens/secret/prompt values travel only through private stdin replies.
        # Code on argv is public worker code; it contains no runtime credentials.
        proc = subprocess.Popen(
            [sys.executable, "-u", "-c", _F12_WORKER_SOURCE],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            text=True, bufsize=1,
        )
        proc.stdin.write(json.dumps({"task": task, "source": source, "config": config}) + "\n")
        proc.stdin.flush()
        for line in proc.stdout:
            try:
                frame = json.loads(line)
            except (ValueError, TypeError):
                continue  # Never relay raw library output, including native-library diagnostics.
            if not isinstance(frame, dict):
                continue
            if "rpc" in frame:
                answer = _f12_bridge_answer(frame)
                proc.stdin.write(json.dumps(answer) + "\n")
                proc.stdin.flush()
                answer = None
            elif "record" in frame:
                try:
                    clean = safe_records(json.dumps([frame["record"]]))[0]
                except Exception:
                    continue
                observed.append(clean)
                _f12_record(clean)
        code = proc.wait()
        _f12_record({"stage": "child_exit", "task": task, "exit_code": code,
                     "fresh_interpreter": True})
        if task == "after_install":
            return code == 0 and any(
                row.get("stage") == "after_install" and row.get("versions_match") is True
                and row.get("shape") == "0.9.4"
                and row.get("loaded_versions", {}).get("numpy", "").split(".")[0] == "2"
                for row in observed)
        return code == 0
    except Exception as error:
        _f12_record({"stage": task, "result": "failed", "error_type": type(error).__name__})
        return False
    finally:
        if proc is not None:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
            proc.stdin.close()
            proc.stdout.close()


In [ ]:
# Uses local wheel paths; no notebook-resources REST API is assumed.
import base64, hashlib, pathlib, subprocess, sys
F12_INSTALL_OK = False
try:
    wheel_dir = pathlib.Path("/tmp/f12-wheels")
    wheel_dir.mkdir(exist_ok=True)
    expected = [{'name': 'sqllocks_shape-0.9.4-py3-none-any.whl', 'bytes': 2991160, 'sha256': '5a327f8c1de7e94e8dfd6c18b479bd8cdfee0889499c6dd409d4bff2500a586c'}, {'name': 'sqllocks_shape_eventhubs-0.9.4-py3-none-any.whl', 'bytes': 22908, 'sha256': '99c204d5296f1a9dd541533098b174edaf2c5001718d1bb7bd84d267aaba095b'}, {'name': 'sqllocks_shape_fabric-0.9.4-py3-none-any.whl', 'bytes': 190690, 'sha256': '5d35d582c6063c6d3df895776efeb79b9e02f671eb8257cc20251705107394b0'}]
    paths = []
    for item in expected:
        name = item["name"]
        path = wheel_dir / name
        if F12_WHEEL_PAYLOADS:
            data = base64.b64decode(F12_WHEEL_PAYLOADS[name], validate=True)
        else:
            # Interactive fallback only: upload unchanged wheels to builtin.
            data = (pathlib.Path("builtin") / name).read_bytes()
        if hashlib.sha256(data).hexdigest() != item["sha256"]:
            raise ValueError("WheelHashMismatch")
        path.write_bytes(data)
        paths.append(str(path))
    # Same Python-kernel pip installation as inline %pip; capture diagnostics privately.
    completed = subprocess.run([sys.executable, "-m", "pip", "install",
        "numpy>=2,<3", "pandas>=2.2.2,<3", *paths,
        "semantic-link-sempy>=0.12", "ipywidgets", "pytest"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    F12_INSTALL_OK = completed.returncode == 0
    _f12_record({"stage":"wheel_install", "hashes_verified":True,
                 "exit_code":completed.returncode, "installed":F12_INSTALL_OK})
except Exception as exc:
    _f12_record({"stage":"wheel_install", "installed":False,"error_type":type(exc).__name__})


In [ ]:
if F12_INSTALL_OK:
    F12_INSTALL_OK = _f12_child("after_install", 'if not F12_INSTALL_OK:\n    _f12_record({"stage":"after_install","result":"skipped"})\nelse:\n    try:\n        import json, sys, importlib.metadata\n        import shape, numpy, pandas, pyarrow\n        from shape.kernel.dispatch import get_kernel\n        loaded_versions = {"numpy":numpy.__version__,"pandas":pandas.__version__,"pyarrow":pyarrow.__version__}\n        versions_match = all(importlib.metadata.version(name)==version for name,version in loaded_versions.items())\n        _f12_record({"stage":"after_install", "python":sys.version.split()[0],"shape":shape.__version__,"kernel":get_kernel().NAME,"imports":"passed","loaded_versions":loaded_versions,"versions_match":versions_match,"packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow","semantic-link-sempy")}})\n        F12_INSTALL_OK = versions_match and shape.__version__ == "0.9.4" and numpy.__version__.split(".")[0] == "2"\n    except Exception as exc:\n        F12_INSTALL_OK = False\n        _f12_record({"stage":"after_install","result":"failed","error_type":type(exc).__name__})\n')
else:
    _f12_record({"stage":"after_install","result":"skipped"})


In [ ]:
if F12_INSTALL_OK:
    _f12_child('memory', '"""Run in a fresh default Fabric Python notebook after the documented wheel install.\n\nRun this file once per row count in a NEW child process so profiling peak RSS excludes\nparent generation and previous measurements. Sizes follow the public plan\'s 5M/50M rows;\nthis is a Shape-generated ten-column synthetic workload, not the D3 benchmark fixture.\n"""\nimport json\nimport os\nimport pathlib\nimport resource\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nimport pyarrow.parquet as pq\nfrom shape.generation.engine import Engine\nfrom shape.generation.schema import Column, GenSchema, Model, Table\n\n\ndef run(rows):\n    columns = {"event_id": Column("event_id", "integer", {"strategy": "sequence"})}\n    for i in range(1, 8):\n        columns[f"value_{i}"] = Column(\n            f"value_{i}", "float", {"strategy": "uniform", "low": 0, "high": 1000}\n        )\n    for i in range(8, 10):\n        columns[f"category_{i}"] = Column(\n            f"category_{i}", "string", {"strategy": "choice", "values": ["a", "b", "c"]}\n        )\n    schema = GenSchema(Model("f12", "f12", seed=12), {"events": Table("events", columns)})\n    engine = Engine(schema, row_counts={"events": rows}, chunk_rows=100_000)\n    with tempfile.TemporaryDirectory(prefix="f12-") as folder:\n        path = pathlib.Path(folder) / "events.parquet"\n        started = time.perf_counter()\n        writer = None\n        try:\n            for batch in engine.iter_chunks("events"):\n                if writer is None:\n                    writer = pq.ParquetWriter(path, batch.schema)\n                writer.write_batch(batch)\n        finally:\n            if writer is not None:\n                writer.close()\n        print(json.dumps({"stage": "generation", "rows": rows, "columns": 10,\n                          "seconds": time.perf_counter() - started,\n                          "parquet_bytes": path.stat().st_size}), flush=True)\n        child = r\'\'\'\nimport json, resource, sys, time\nfrom shape.profile.engine import EngineOptions, profile_table\nfrom shape.kernel.dispatch import get_kernel\nstart = time.perf_counter()\nprofile = profile_table(sys.argv[1], options=EngineOptions(mode="bounded", threads=2))\nprint(json.dumps({"stage":"profile", "mode":"bounded", "kernel":get_kernel().NAME,\n "rows":profile["rows"], "columns":len(profile["columns"]),\n "seconds":time.perf_counter()-start,\n "peak_rss_raw":resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n "peak_rss_unit":"KiB" if sys.platform.startswith("linux") else "bytes"}))\n\'\'\'\n        result = subprocess.run([sys.executable, "-c", child, str(path)], capture_output=True, text=True)\n        print(result.stdout, end="", flush=True)\n        print(json.dumps({"stage": "child_exit", "rows": rows, "exit_code": result.returncode}), flush=True)\n        if result.returncode:\n            # Do not print unfiltered exception text that could contain identity information.\n            print(json.dumps({"stage": "error", "rows": rows,\n                              "error_type": result.stderr.strip().splitlines()[-1].split(":")[0]\n                              if result.stderr.strip() else "no_stderr"}), flush=True)\n\n\nfor count in (5_000_000, 50_000_000):\n    output = io.StringIO()\n    try:\n        with contextlib.redirect_stdout(output):\n            run(count)\n        for line in output.getvalue().splitlines():\n            try: value = json.loads(line)\n            except (ValueError, TypeError): continue\n            if isinstance(value, dict): _f12_record(value)\n    except Exception as error:\n        _f12_record({"stage":"memory","rows":count,"result":"failed","error_type":type(error).__name__})\n')
else:
    _f12_record({"stage":'memory',"result":"skipped"})


## Semantic-model qualification
This runs in a fresh interpreter with an in-memory bridge to this notebook's own credentials. The original test body's single profile result is reused for the summary.


In [ ]:
if F12_INSTALL_OK:
    _f12_child('semantic_model', 'if not F12_INSTALL_OK:\n    _f12_record({"stage":"semantic_model","result":"skipped"})\nelse:\n    # Uses the existing live test body with the notebook user\'s identity.\n    # The service-principal pytest fixture is deliberately not invoked in this Owner run.\n    import os, json, logging, contextlib, io\n    import pytest\n    os.environ["FABRIC_WORKSPACE_ID"] = "Shape Test"\n    os.environ["FABRIC_SEMANTIC_MODEL"] = "shape_sm"\n    ns = {}\n    captured = io.StringIO()\n    previous_disable = logging.root.manager.disable\n    try:\n        logging.disable(logging.CRITICAL)\n        with contextlib.redirect_stdout(captured), contextlib.redirect_stderr(captured):\n            exec(\'"""The ``semantic-model://`` source and ``shape profile-model`` against a real semantic model\\n(only where the Fabric test settings exist; ``sempy`` must be able to sign in).\\n\\n    FABRIC_WORKSPACE_ID        the workspace (name or GUID) that holds the model\\n    FABRIC_SEMANTIC_MODEL      the model (name or GUID); the live test profiles all of it\\n\\n    pytest -m live plugins/shape-fabric/tests/test_live_semantic_model.py\\n\\nA missing variable fails the test with the variable\\\'s name; service failures also fail.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport os\\n\\nimport pytest\\nfrom shape_fabric import semantic_source as ss\\nfrom shape_fabric.semantic_profile import profile_model\\n\\npytestmark = pytest.mark.live\\n\\n\\ndef need(name):\\n    value = (os.environ.get(name) or "").strip()  # the variable has held a trailing space\\n    if not value:\\n        pytest.fail(f"live test needs {name}")\\n    return value\\n\\n\\n@pytest.fixture\\ndef service_principal():\\n    """sempy signs in as the service principal (semantic-link >= 0.12 ``set_service_principal``,\\n    the documented way outside a notebook), not as a notebook user."""\\n    tenant, client, secret = (\\n        need("FABRIC_TENANT_ID"),\\n        need("FABRIC_CLIENT_ID"),\\n        need("FABRIC_CLIENT_SECRET"),\\n    )\\n    try:\\n        from sempy.fabric import set_service_principal\\n    except ImportError as exc:\\n        pytest.fail(f"live test needs semantic-link-sempy >= 0.12 ({exc})")\\n    with set_service_principal(tenant, client, client_secret=secret):\\n        yield\\n\\n\\ndef test_a_real_model_profiles_with_its_relationships_and_every_table_reads(service_principal):\\n    workspace, model = need("FABRIC_WORKSPACE_ID"), need("FABRIC_SEMANTIC_MODEL")\\n    names = ss.table_names(workspace, model)\\n    assert names, "the model has no tables"\\n    data = profile_model(workspace, model, max_rows=1000).to_dict()\\n    assert list(data["tables"]) == names\\n    for r in data["relationships"]:\\n        assert r["evidence"] == "declared" and r["source"] == "semantic model"\\n    first = names[0]\\n    source = ss.SemanticModelSource()\\n    uri = ss.build(workspace, model, first)\\n    schema = source.schema(uri)\\n    batches = list(source.read(uri, max_rows=10))\\n    assert sum(b.num_rows for b in batches) <= 10\\n    assert all(b.schema.equals(schema) for b in batches)\\n\', ns)\n            original_profile_model = ns["profile_model"]\n            captured_profiles = []\n            def capture_profile(*args, **kwargs):\n                result = original_profile_model(*args, **kwargs)\n                captured_profiles.append(result)\n                return result\n            ns["profile_model"] = capture_profile\n            try:\n                ns["test_a_real_model_profiles_with_its_relationships_and_every_table_reads"](None)\n            finally:\n                ns["profile_model"] = original_profile_model\n            if len(captured_profiles) != 1:\n                raise RuntimeError("UnexpectedProfileCount")\n            data = captured_profiles[0].to_dict()\n        _f12_record({"stage":"semantic_model", "test_body":"passed", "identity":"current notebook identity", "tables":[{"table_index":i,"rows":t.get("row_count"),"columns":len(t.get("columns",{}))} for i,t in enumerate(data["tables"].values())],"relationships":len(data.get("relationships",[]))})\n    except (Exception, pytest.fail.Exception) as exc:\n        _f12_record({"stage":"semantic_model", "test_body":"failed", "error_type":type(exc).__name__})\n    finally:\n        logging.disable(previous_disable)\n')
else:
    _f12_record({"stage":'semantic_model',"result":"skipped"})


## Eventstream qualification
This runs in a fresh interpreter, preserving every readiness, source-binding and item guard. Parent notebook credentials and hidden prompts are bridged through private pipes, never files, environment variables or argv.


In [ ]:
if F12_INSTALL_OK:
    _f12_child('events', '# API/Key Vault in unattended jobs; hidden prompts only in interactive fallback.\nimport getpass, time, urllib.parse, warnings\nnotebookutils = _f12_notebook_bridge\nconnection = host = database = None\nobserved_destination_status = "NotChecked"\ntry:\n    if not F12_INSTALL_OK: raise RuntimeError("InstallUnavailable")\n    with _f12_silent():\n        import sempy.fabric as fabric\n        notebookutils = _f12_notebook_bridge\n        import pyarrow as pa\n        from shape_fabric.eventstream import EventstreamEmitter\n        from shape_fabric.eventstream_writer import EventstreamWriter\n        from shape_fabric.eventhouse_writer import EventhouseWriter\n        from shape.streaming.emit.formats import with_event_fields\n        api = fabric.FabricRestClient()\n        prefix = "v1/workspaces/e90940bd-3f32-42ce-998e-ef2ddc74355b/"\n        def get(path):\n            response = api.get(prefix + path)\n            if response.status_code != 200: raise RuntimeError("FabricReadFailed")\n            return response.json()\n        # Interactive fallback may discover only the specifically named f12 items.\n        if not F12_EVENTSTREAM_ID or not F12_KQL_DATABASE_ID:\n            if F12_NONINTERACTIVE: raise RuntimeError("MissingItemReferences")\n            items = get("items")["value"]\n            F12_EVENTSTREAM_ID = next(x["id"] for x in items if x["displayName"]=="f12-eventstream" and x["type"]=="Eventstream")\n            F12_KQL_DATABASE_ID = next(x["id"] for x in items if x["displayName"]=="f12-kql" and x["type"]=="KQLDatabase")\n        es = get("items/"+F12_EVENTSTREAM_ID)\n        db = get("kqlDatabases/"+F12_KQL_DATABASE_ID)\n        if es["displayName"]!="f12-eventstream" or not db["displayName"].startswith("f12-"):\n            raise RuntimeError("ItemNameGuard")\n        topology = get("eventstreams/"+F12_EVENTSTREAM_ID+"/topology")\n        source = next(x for x in topology["sources"] if x["name"]=="f12-input" and x["type"]=="CustomEndpoint")\n        destination = next(x for x in topology["destinations"] if x["name"]=="f12-destination" and x["type"]=="Eventhouse")\n        ready = get("eventstreams/"+F12_EVENTSTREAM_ID+"/destinations/"+destination["id"])\n        current_status = ready.get("status")\n        observed_destination_status = current_status if current_status in ("Running","Warning","Creating","Stopped","Paused","Starting","Failed") else "Other"\n        properties = ready["properties"]\n        if ready.get("status")!="Running" or ready.get("error"):\n            raise RuntimeError("DestinationNotRunning")\n        if properties.get("dataIngestionMode")!="ProcessedIngestion" or properties.get("itemId")!=F12_KQL_DATABASE_ID:\n            raise RuntimeError("DestinationConfigurationMismatch")\n        if properties.get("tableName")!="f12-events" or properties.get("inputSerialization",{}).get("type")!="Json":\n            raise RuntimeError("DestinationConfigurationMismatch")\n        if source.get("status")!="Running": raise RuntimeError("SourceNotRunning")\n        # Connection metadata binds credentials to this exact newly created source.\n        endpoint = None\n        try:\n            endpoint = get("eventstreams/"+F12_EVENTSTREAM_ID+"/sources/"+source["id"]+"/connection")\n        except Exception:\n            pass  # Never expose a response body or exception containing SAS secrets.\n        expected_namespace = expected_hub = None\n        if F12_VAULT_URL and F12_SECRET_NAME:\n            bundle = json.loads(notebookutils.credentials.getSecret(F12_VAULT_URL,F12_SECRET_NAME))\n            if bundle.get("eventstream_id") != F12_EVENTSTREAM_ID or bundle.get("source_id") != source["id"]:\n                raise RuntimeError("SecretSourceMismatch")\n            connection, host, database = (bundle[k] for k in ("connection_string","host","database"))\n            expected_namespace, expected_hub = (bundle[k] for k in ("namespace","event_hub_name"))\n        elif endpoint:\n            connection = endpoint["accessKeys"]["primaryConnectionString"]\n            if "EntityPath=" not in connection:\n                connection += ";EntityPath="+endpoint["eventHubName"]\n            host = urllib.parse.urlsplit(db["properties"]["queryServiceUri"]).hostname\n            database = db["displayName"]\n        elif F12_NONINTERACTIVE:\n            raise RuntimeError("ConnectionUnavailable")\n        if endpoint:\n            # API metadata takes precedence over manually supplied bundle metadata.\n            expected_namespace = endpoint["fullyQualifiedNamespace"]\n            expected_hub = endpoint["eventHubName"]\n    # Prompts are outside captured output so the Owner can see labels, never values.\n    if not connection:\n        with warnings.catch_warnings():\n            warnings.simplefilter("error", getpass.GetPassWarning)\n            connection = getpass.getpass("f12-input connection string (hidden): ")\n            host = getpass.getpass("New f12 Eventhouse Query URI host (hidden): ").strip()\n            database = getpass.getpass("New f12 KQL database name (hidden): ").strip()\n            expected_namespace = getpass.getpass("Namespace copied from this f12-input Details (hidden): ").strip()\n            expected_hub = getpass.getpass("Event hub name copied from this f12-input Details (hidden): ").strip()\n    pairs = [entry.split("=",1) for entry in connection.split(";") if entry]\n    fields = {key.lower():value for key,value in pairs}\n    if len(fields) != len(pairs): raise RuntimeError("DuplicateConnectionFields")\n    if urllib.parse.urlsplit(fields["endpoint"]).hostname != expected_namespace.lower() or fields.get("entitypath") != expected_hub:\n        raise RuntimeError("ConnectionEndpointMismatch")\n    assert database.startswith("f12-"), "Use only the new f12 database"\n    if database != db["displayName"]: raise RuntimeError("DatabaseMismatch")\n    if host != urllib.parse.urlsplit(db["properties"]["queryServiceUri"]).hostname:\n        raise RuntimeError("HostMismatch")\n    _f12_record({"stage":"destination_readiness","source_running":True,"destination_running":True,"warning":False})\n    with _f12_silent():\n        batch = pa.record_batch({"id":list(range(100)),"marker":["f12-live"]*100})\n        emitter = EventstreamEmitter()\n        emitted = emitter.emit("eventstream://f12-eventstream",[with_event_fields(batch,"f12-emitter",0)],connection_string=connection)\n        emitter.close()\n        writer = EventstreamWriter("eventstream://f12-eventstream",connection_string=connection)\n        written = writer.write_table("f12-writer",[batch]); writer.close()\n        direct = EventhouseWriter(f"eventhouse://{host}/{database}",token=lambda:notebookutils.credentials.getToken("kusto"))\n        accepted = direct.write_table("f12-direct",[batch])\n    _f12_record({"stage":"events_sent","emitter_acknowledged":emitted,"stream_writer_acknowledged":written,"direct_writer_accepted":accepted})\n    poll_started=time.monotonic()\n    for attempt in range(30):\n        with _f12_silent():\n            raw=direct.client.query("[\'f12-events\'] | where marker == \'f12-live\' | summarize raw_rows=count() by _shape_table")\n            deduped=direct.client.query("[\'f12-events\'] | where marker == \'f12-live\' | summarize by _shape_table, _shape_seq | summarize unique_rows=count() by _shape_table")\n            direct_count=direct.client.query("[\'f12-direct\'] | count")\n        counts_match=sorted(deduped)==[["f12-emitter",100],["f12-writer",100]] and direct_count==[[100]]\n        if counts_match or attempt==29: break\n        time.sleep(10)\n    _f12_record({"stage":"events_landed","raw_rows_by_table":raw,"unique_rows_by_table":deduped,"direct_writer_count":direct_count,"counts_match":counts_match,"poll_attempts":attempt+1,"poll_seconds":time.monotonic()-poll_started})\nexcept Exception as exc:\n    _f12_record({"stage":"events","result":"failed","error_type":type(exc).__name__,"destination_status":observed_destination_status})\nfinally:\n    connection=host=database=None\n')
else:
    _f12_record({"stage":'events',"result":"skipped"})
if F12_NONINTERACTIVE:
    import notebookutils
    notebookutils.notebook.exit(json.dumps(_F12_RECORDS))
